![SAAI](https://i.imgur.com/YHRhgqv.png)

[English](data_exploration_and_preparation.ipynb) | **العربية**

# فرصة: جهّز بيانات الإعلانات وافهمها

**هدفنا:** تحويل الإعلانات الخام إلى جدول يمكن فحصه وحفظه مع إبقاء الدليل الأصلي. سنتابع الإعلانات التدريبية الخيالية العشرة نفسها ثم نعيد استخدام المسار على الألف كاملة.

**المسار:** قراءة ← استكشاف ← تنظيف ← استخراج ← فحص المجهول ← مراجعة المصادر المرتبطة ← مراجعة المواعيد ← معالجة الجميع ← حفظ.

لدينا **تسعة تمارين صغيرة**، يحتاج كل منها إلى 1–3 عبارات. قبل كل تمرين ستعرف عمل العملية وسبب أهميتها وترى مثالًا يعمل. استبدل `...` قبل تشغيل الخلية، وشغّل الخلايا بالترتيب.

يتولى ملف [preparation_helpers.py](preparation_helpers.py) المقدم الاستخراج المتكرر ومعالجة التواريخ والحفظ. لا تحتاج إلى تعديله لإنهاء الدرس. يمكنك فحص كوده، وسنشرح مدخلاته ومخرجاته وحدوده هنا. مهمتك استخدام المسار بوعي وفحص نتائجه، وليس بناء كل آلية من الصفر.

**الناتج:** صف مجهّز لكل مصدر يتضمن حقائق محددة والنص الأصلي والمجهول وعلامات المراجعة. هذا أساس للتجهيز وليس قاعدة فرص مكتملة أو خلفية توصيات.

## الإعداد — شغّل الخلايا المقدمة

استخدم Python 3.11 أو أحدث في Jupyter. افتح الدفتر من مجلد `data/` وأبقِ `preparation_helpers.py` بجانبه. إذا تعذر الاستيراد أو العثور على ملف، افحص `Path.cwd()` والنواة المختارة.

توفر **pandas** جدول `DataFrame` بصفوف وأعمدة. تعرض `display()` الجدول، وتتعامل `Path` مع المسارات. تكفي هذه الأدوات للدرس؛ لا نحتاج تنزيل نموذج أو مفتاح API.

إذا كانت الحزم ناقصة فأزل التعليق عن `%pip` أدناه. يثبتها في بيئة الدفتر، وتمسح `clear_output()` سجل التثبيت. أعد تشغيل النواة إذا طُلب ذلك.

In [ ]:
# Uncomment if needed / أزل التعليق عند الحاجة
# %pip install pandas ipython
from IPython.display import clear_output
clear_output(wait=False)

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display
from preparation_helpers import (
    extract_facts, related_sources, review_deadlines,
    prepare_announcements, save_prepared_data,
)
DATA_PATH = Path("raw/raw_opportunities.csv")

## 1. اقرأ إعلانًا

يحتوي CSV على **1,000 إعلان عربي تخيلي**: بريد ومنشورات اجتماعية ونصوص كتيبات ورسائل مجموعات ولوحات إعلانات. الصف إعلان وليس بالضرورة فرصة مستقلة. توقع إعادة نشر وتحديثات وحقائق ناقصة واختلافات.

| العمود | المعنى |
|---|---|
| `source_id` | معرّف المصدر؛ احتفظ به |
| `source_type` | صيغة الإعلان |
| `source_name` | الناشر |
| `published_at` | وقت النشر وليس موعد تقديم أو وقت تحقق |
| `content` | النص الأصلي للإعلان |

تقرأ `pd.read_csv()` ملف CSV بما فيه الفواصل والأسطر داخل الخانات المقتبسة. يحفظ `dtype="string"` الحقول كنص، وتقرأ `utf-8-sig` ترميز الملف. تعرض `.head(3)` أول ثلاثة صفوف.

**فكرتان قبل البدء:** الإعلان صف مصدر واحد، أما الفرصة فهي النشاط الذي قد تصفه عدة مصادر. تجهيز ألف إعلان لا يعني الحصول على ألف فرصة مختلفة. سنحفظ المصدر أولًا ثم يقرر فريقك كيف يربط المصادر بالفرص.

In [ ]:
raw = pd.read_csv(DATA_PATH, encoding="utf-8-sig", dtype="string")
print("Sources:", len(raw))
display(raw.head(3))

تختار `raw.iloc[0]` الصف الأول، وتقرأ `row["content"]` نصه. اقرأ إعلانًا كاملًا قبل تحديد المعلومات المستخرجة.

In [ ]:
row = raw.iloc[0]
print(row["source_id"])
print(row["content"])

### دورك · اقرأ المصدر التالي

اختر الصف الثاني باستخدام `.iloc[1]` واطبع نصه. **عبارتان.** المتوقع `SRC-0002`، بريد يذكر 50 ريالًا و4 أكتوبر 2026.

In [ ]:
second = ...
print(...)

## 2. استكشف ثم اعمل على عينة صغيرة

يساعد **الاستكشاف** على ملاحظة اختلاف الصيغ قبل معالجة الجميع. يختار `raw["source_type"]` عمودًا وتحسب `.value_counts()` فئاته. تعدّ `.isna().sum()` خانات CSV الناقصة. قد يفتقد نص غير فارغ الرسوم أو الموعد.

تنشئ `.head(5).copy()` عينة مستقلة من خمسة صفوف. نستخدم نسخة ليبقى الجدول الأصلي دون تغيير.

In [ ]:
display(raw["source_type"].value_counts())
display(raw.isna().sum())
example_sample = raw.head(5).copy()
print("Example sample:", len(example_sample))

### دورك · اختر عشرة مصادر

اتبع مثال العينة وغيّر 5 إلى 10. **عبارة واحدة.** المتوقع عشرة صفوف تتضمن صيغًا مختلفة وزوج مصادر مرتبطًا.

In [ ]:
sample = ...
display(sample[["source_id", "source_type"]])

تختار الأقواس المزدوجة أعلاه عمودين للعرض. سنستخدم `sample` نفسها في الخطوات التالية ليبني كل قسم على سابقه.

## 3. نظّف التنسيق واحفظ المعنى

يصلح التنظيف التنسيق لا الحقائق. حافظ على الأرقام والتواريخ والأسطر والنفي مثل «لا». تزيل `.str.strip()` المسافات المحيطة فقط لكل نص في العمود. احفظ النتيجة في عمود جديد `cleaned_content` واترك `content` كما هو.

هذا مثال مكتمل. عدم تغير بعض الصفوف طبيعي؛ لا تضف تنظيفًا دون مشكلة لاحظتها.

In [ ]:
example_text = pd.Series(["  لا تتطلب خبرة. الرسوم 100 ريال.  "])
display(example_text.str.strip())

### دورك · نظّف العينة

طبّق `.str.strip()` على `sample["content"]` كما في المثال. **عبارة واحدة.** قارن العمودين؛ يجب ألا تتغير الكلمات والمبالغ.

In [ ]:
sample["cleaned_content"] = ...
display(sample[["content", "cleaned_content"]].head(2))

## 4. استخرج الحقائق وافحص الأدلة

يحوّل **الاستخراج** الصياغة إلى حقول، ويمنح **التوحيد** الحقيقة المعروفة تمثيلًا متسقًا مثل «المشاركة مجانية» ← `0`. الدليل هو نص المصدر الذي يدعم القيمة.

تأخذ الدالة المقدمة `extract_facts(table)` صفوف مصادر منظفة وتعيد نسخة بالحقول التالية:

| الحقل | المعنى |
|---|---|
| `title_candidates` | العبارات داخل علامات الاقتباس العربية |
| `title_suggestion` | اقتراح عند وجود عبارة واحدة فقط |
| `title_needs_review` | لا يوجد مرشح واحد واضح؛ افحص المصدر |
| `fee_evidence` | عبارة الرسوم المطابقة محفوظة للمراجعة |
| `cost_sar` | مبلغ رقمي أو صفر للمجاني الصريح أو مجهول |
| `activity_url` | رابط نشاط محدد تم التعرف عليه |

**الحدود:** يتعرف المساعد على صيغ ملحوظة قليلة لا على كل العربية، وقد يفوّت حقائق. قد تشمل العبارات المقتبسة اسم الناشر والعنوان معًا. يتطلب النفي أو تعدد ادعاءات الرسوم أو التحديثات قراءة المصدر؛ أول تطابق ليس حقيقة مضمونة. لا تؤكد الدالة الأهلية ولا تدمج الإعلانات.

يمكنك فحص [تنفيذ المساعد](preparation_helpers.py)، لكنك لا تحتاج كتابة أنماطه النصية. شاهد عمله على مصدر واحد أولًا.

**انتبه للتطابق الجزئي:** `تحدي` يظهر داخل `تحديد`؛ لا يعني ذلك أن النشاط مسابقة. افحص سياق أمثلة متنوعة قبل توسيع قواعدك. الأدوات المقدمة محدودة وليست محللًا شاملًا.

In [ ]:
one_prepared = extract_facts(sample.head(1))
display(one_prepared[["source_id", "title_suggestion", "fee_evidence", "cost_sar"]])

### دورك · جهّز المصادر العشرة

استدعِ `extract_facts()` مع `sample` وفق المثال. **عبارة واحدة.** المتوقع عشرة صفوف؛ المصدر 1 مجاني والمصدر 2 برسوم 50 ريالًا.

In [ ]:
sample_facts = ...
display(sample_facts[["source_id", "title_suggestion", "fee_evidence", "cost_sar"]])

## 5. المجهول يختلف عن المجاني

تعرض pandas المجهول كـ `<NA>` أو `NaN` ويحفظه JSON كـ `null`. **لا تستبدل الرسوم الناقصة بصفر.** يذكر المصدر 4 أن الرسوم غير محددة، لذا تبقى مجهولة.

يختار **الترشيح** الصفوف المطابقة. تحدد `.isna()` القيم الناقصة وتعرض `.loc[condition, columns]` الصفوف والأعمدة المطلوبة فقط. يفحص المثال المكتمل التالي التباس العنوان؛ وفر المساعد علامة `title_needs_review` المنطقية مسبقًا.

In [ ]:
needs_title_review = sample_facts["title_needs_review"]
display(sample_facts.loc[needs_title_review, ["source_id", "title_candidates"]])

### دورك · اعثر على الرسوم المجهولة

استخدم `.isna()` على `sample_facts["cost_sar"]` لإكمال الشرط. **عبارة واحدة.** المتوقع وجود المصدر 4 ضمن النتائج. اقرأ نصه لتشرح لماذا المجهول صحيح.

In [ ]:
missing_fee = ...
display(sample_facts.loc[missing_fee, ["source_id", "content", "cost_sar"]])

**افصل المعاني:** «يشترط إتقان SQL» شرط إلزامي، و«يفضل معرفة Excel» تفضيل. غياب المتطلبات لا يعني عدم وجودها. لا يستخرج مثالنا المقدم هذه الحقول؛ يختار فريقك كيفية توسيع التجهيز لتطبيقه.

## 6. المصادر المرتبطة تحتاج مراجعة

**إعادة النشر** تكرر الإعلان. **التحديث الصريح** يذكر المعلومة التي تغيرت. **التعارض** يقدم ادعاءات غير متوافقة دون حسم مبرر.

تعيد `related_sources(table)` صفوف المصادر التي تشترك في رابط نشاط غير ناقص. **لا** تدمج أو تحذف أو تختار الادعاء الصحيح. لا تعد الروابط الناقصة هوية مشتركة. قارن الجهة والتواريخ والصياغة قبل تحديد إن كانت المصادر تصف النشاط نفسه.

مثلًا، يكون العرض التالي فارغًا لأول خمسة صفوف. فراغ قائمة المراجعة نتيجة صحيحة ممكنة.

الرابط المشترك علامة للمراجعة وليس إثباتًا للهوية. إذا اختلف الناشر، لا تعتبر أحدث نص تحديثًا معتمدًا تلقائيًا. احتفظ بالمصدرين حتى تراجع العلاقة والأدلة.

In [ ]:
display(related_sources(sample_facts.head(5)))

### دورك · اعثر على الزوج المرتبط

استدعِ المساعد نفسه على `sample_facts` كاملة. **عبارة واحدة.** المتوقع `SRC-0006` و`SRC-0007` وكلاهما يذكر 120 ريالًا. اقرأ النصين؛ يقول المصدر 7 «إعادة نشر» صراحة. لا تحذف أي مصدر.

In [ ]:
review_pair = ...
display(review_pair[["source_id", "source_name", "cost_sar", "content"]])

**نقطة قرار:** لا يذكر المصدر 7 بعض حقائق المصدر 6. الغياب لا يلغي الدليل السابق. لو ذكر مصدر آخر 150 ريالًا فاحتفظ بالادعاءين للمراجعة. يغيّر تحديث الموعد الموثوق هذا الحقل فقط ولا يستبدل غيره. التشابه وحده ليس إذنًا بحذف مصدر.

**مثال توضيحي، وليس إجابة من البيانات:** النشاط الأصلي من 20 إلى 22 أكتوبر. يقول التحديث «انتقلت البداية إلى 25 أكتوبر» ولا يذكر النهاية. البداية الجديدة 25 أكتوبر؛ النهاية والمدة الجديدتان مجهولتان. لا تحرّك النهاية إلى 27 أكتوبر بالتخمين. لا توجد شيفرة إضافية مطلوبة هنا؛ اشرح السبب لزميلك.

## 7. راجع المواعيد عند وقت محدد

**وقت التقييم** هو «الآن» المستخدم للقرار. استخدم **18 أكتوبر 2026 الساعة 12 ظهرًا بتوقيت الرياض** بما يطابق إعداد الخلفية. تمثل `pd.Timestamp()` هذه اللحظة، و`+03:00` فرق المنطقة الزمنية. لا تغيّر تواريخ البيانات.

تقرأ `review_deadlines(table, evaluation_time)` صياغة «آخر يوم للتقديم …» الملحوظة، وتحفظ التاريخ المستخرج وتضيف `passed` أو `not_passed` أو `uncertain` مع وقت التقييم. لا تستخدم وقت النشر كموعد ولا تحسم التحديثات ولا تستنتج الأهلية.

تفتقد التواريخ ساعة ومنطقة الإغلاق. سياسة المراجعة المقدمة متحفظة عمدًا: **قبل 17 أكتوبر ← انتهى؛ بعد 19 أكتوبر ← لم ينتهِ؛ من 17 إلى 19 أكتوبر أو ناقص أو غير صالح ← غير مؤكد**. يمنع هامش المراجعة افتراض منطقة زمنية غير مذكورة. ليس هذا محلّل موعد إغلاق دقيق؛ لو كانت لحظة الإغلاق معلومة لانتهى الموعد عند بلوغها. يطبق المساعد الهامش المقابل عند تغيير وقت التقييم.

الموعد القادم وحده لا يثبت فتح التسجيل. نحتفظ بجميع المصادر بما فيها المنتهية.

**حالة المصدر ليست الحالة الحالية:** إذا قال إعلان إن التسجيل لم يبدأ وحدد يومًا مضى، فلا تستنتج أنه مفتوح الآن. احتفظ بالدليل واشرح أن الحالة تحتاج تحققًا. لا يوجد تمرين برمجي إضافي هنا.

In [ ]:
EVALUATION_TIME = pd.Timestamp("2026-10-18T12:00:00+03:00")
sample_prepared = review_deadlines(sample_facts, EVALUATION_TIME)
display(sample_prepared[["source_id", "deadline_date", "deadline_state"]])

كما في القسم 5، رشّح عرضًا للفحص. تعني `!=` «لا يساوي». يحسب المثال الحالات، وينشئ تمرينك عرض مرشحين دون حذف صفوف من `sample_prepared`.

In [ ]:
display(sample_prepared["deadline_state"].value_counts())

### دورك · احتفظ بعرض مرشحين منفصل

أكمل الشرط بـ `sample_prepared["deadline_state"] != "passed"`. **عبارة واحدة.** يستبعد العرض المصدر 2، ويبقى المصدر 1 غير مؤكد لا متاحًا بشكل مؤكد. تبقى الصفوف العشرة الأصلية.

In [ ]:
candidates = sample_prepared.loc[...].copy()
display(candidates[["source_id", "deadline_state"]])

## 8. أعد استخدام المسار على جميع المصادر

تابعنا التنظيف ثم الاستخراج ثم مراجعة الموعد. تنفذ `prepare_announcements(table, evaluation_time)` **الخطوات الثلاث نفسها** بالترتيب. تعيد صفًا لكل مصدر وتحفظ النص الأصلي وتبقي المجهول ظاهرًا. لا تدمج السجلات ولا تحسم التعارضات ولا تولّد تضمينات أو حقائق غير مذكورة.

هذه دالة مقدمة لإعادة الاستخدام وليست خوارزمية جديدة للتعلم. شغّلها أولًا على 50 مصدرًا وافحص نتائج قليلة ثم استخدم الاستدعاء نفسه للجدول كاملًا. لا تحتاج حلقة إضافية.

In [ ]:
prepared_fifty = prepare_announcements(raw.head(50), EVALUATION_TIME)
print("Prepared sources:", len(prepared_fifty))
display(prepared_fifty[["source_id", "cost_sar", "deadline_state"]].head(3))

### دورك · جهّز المصادر الألف

كرر المثال مع `raw` بدل `raw.head(50)`. **عبارة واحدة.** المتوقع ألف صف مع بقاء الرسوم المجهولة والتواريخ غير المؤكدة.

In [ ]:
prepared = ...
print("Prepared sources:", len(prepared))
display(prepared["deadline_state"].value_counts())

**تحقق من الحفظ:** تعدّ `.nunique()` المعرفات المختلفة، وتقارن `.equals()` الأعمدة مع ترتيبها. يجب أن تعرض الفحوص المقدمة `1000` ثم `True` ثم `True`. تتحقق من حفظ المصادر لا دقة الاستخراج؛ استمر في فحص أدلة الحقائق الملتبسة والناقصة.

In [ ]:
print("Distinct IDs:", prepared["source_id"].nunique())
print("IDs preserved:", prepared["source_id"].equals(raw["source_id"]))
print("Original text preserved:", prepared["content"].equals(raw["content"]))

## 9. احفظ وأعد التحميل واشرح قراراتك

يحفظ **JSON** السجلات في ملف. يصبح المجهول `null` وتبقى العربية مقروءة. تنشئ `save_prepared_data(table, path)` مجلد الوجهة وتكتب JSON وتعيد مساره. تستبدل إعادة التشغيل ملف JSON المختار ولا تغيّر CSV الخام.

يحفظ المثال عينة العشرة فقط في `prepared/lesson_sample.json`. يستخدم تمرينك مسارًا آخر للنتيجة الكاملة. لا تكتب صادرات التدريب فوق مخزن فرص التطبيق. ليست التواريخ المحفوظة دليلًا على ساعة إغلاق عند منتصف الليل.

**توقّع شكل الصف:** تبقى `source_id` و`content`، وتضاف مثلًا `fee_evidence` و`cost_sar`. إذا لم تثبت الأداة الرسوم، تبقى القيمة مجهولة. هذا سجل مصدر للمراجعة، وليس استجابة API جاهزة.

In [ ]:
sample_path = save_prepared_data(sample_prepared, Path("prepared/lesson_sample.json"))
print(sample_path)

### دورك · احفظ الجدول المجهز كاملًا

استدعِ `save_prepared_data()` مع `prepared` و`output_path` المقدم كما في المثال. **عبارة واحدة.** المتوقع ملف JSON يتضمن المصادر الألف.

In [ ]:
output_path = Path("prepared/source_preparation.json")
saved_path = ...
print(saved_path)

أخيرًا، تعيد `pd.read_json()` تحميل الملف. تعني `orient="records"` قائمة سجلات صفوف، وتبقي `convert_dates=False` التواريخ كنص للفحص. تحول `.tolist()` الأعمدة إلى قوائم Python عادية لمقارنة القيم المحفوظة والأصلية. المتوقع ألف صف ومقارنتان بقيمة `True`.

In [ ]:
reloaded = pd.read_json(saved_path, orient="records", convert_dates=False)
print("Reloaded sources:", len(reloaded))
print("IDs preserved:", reloaded["source_id"].tolist() == raw["source_id"].tolist())
print("Text preserved:", reloaded["content"].tolist() == raw["content"].tolist())
display(reloaded[["source_id", "cost_sar", "deadline_state"]].head(3))

## ما أنجزته

حمّلت المصادر وفحصتها ونظفتها وجهزتها وحفظتها باستخدام مسار مقدم. راجعت الرسوم المجهولة والعناوين الملتبسة والمصادر المرتبطة والمواعيد المنتهية أو غير المؤكدة. لم تحتج كتابة أنماط الاستخراج أو آلية التصدير المساندة.

**اشرح ثلاثة قرارات:** لماذا ليست الرسوم الناقصة صفرًا، ولماذا لا يسمح الرابط المشترك بالحذف، ولماذا لا يؤكد الموعد القادم الأهلية. هذه التفسيرات مهمة بقدر إكمال الفراغات.

**الخطوة التالية لفريقك:** وسّع التجهيز لحقول تطبيقك وراجع التحديثات والتعارضات واختر تمثيل الفرص واربط مخزنك عبر [دليل الخلفية](../backend/README.ar.md). جدول التدريب ليس مخطط استجابة API الثابت، ولا نفرض إطارًا أو مخزنًا محددًا.

التضمينات اختيارية لاحقًا؛ تمثل النص رقميًا للبحث بالتشابه ولا تثبت تطابق الإعلانات. ليست مطلوبة لهذا الدرس.

[دليل البيانات](README.ar.md) · [المساعدات المقدمة](preparation_helpers.py)

**أول خطوة بعد الدفتر:** افتح الملف الذي صدّرته، واختر نشاطًا تدعمه مصادر فعلية، ثم اقرأه من مخزنك في `repository.py` وحوّله إلى `OpportunityDetailsResponse`. جرّب معرّفه في `/docs` قبل توسيع البحث. لا يقرأ الخادم مخرجات خلايا الدفتر ولا يشغله تلقائيًا. راجع [خطوات الربط](../backend/README.ar.md#first-working-integration).

قبل كتابة خلفية التطبيق، أكمل [ورقة ربط الحقول](README.ar.md#mapping) لنشاط واحد. حدّد ما يلزم لأول استجابة تفاصيل وما يمكن أن يبقى مجهولًا. إنهاء هذه التمارين لا يعني اكتمال محرك البحث؛ هذه هي مرحلة الربط التالية.

![SAAI](https://i.imgur.com/beBR1m2.png)